# Functions and Mappings - Examples

Interactive examples demonstrating functions, their properties, and ML applications.

**Topics covered:**
1. Function basics and notation
2. Domain, codomain, range
3. Injective, surjective, bijective
4. Function composition
5. Inverse functions
6. ML applications

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Function Basics

A **function** f: A → B maps each element in domain A to exactly one element in codomain B.

| Term | Symbol | Description |
|------|--------|-------------|
| Domain | A | Set of valid inputs |
| Codomain | B | Set of possible outputs |
| Range | f(A) | Actual outputs (⊆ codomain) |
| Image | f(x) | Output for specific input x |

In [ ]:
# Defining functions in Python

# Simple function: f(x) = x²
def square(x):
    return x ** 2

# Domain: all real numbers (in practice, what we pass)
domain = np.array([-2, -1, 0, 1, 2])
range_values = square(domain)

print(f"Domain: {domain}")
print(f"Range:  {range_values}")
print(f"\nNote: Range {set(range_values)} ⊆ Codomain (all non-negative reals)")

## 2. Function Properties: Injective, Surjective, Bijective

| Property | Definition | ML Example |
|----------|------------|------------|
| **Injective** (1-to-1) | f(a) = f(b) ⟹ a = b | Embedding layers |
| **Surjective** (onto) | ∀y∈B, ∃x: f(x)=y | Softmax (onto simplex) |
| **Bijective** | Both injective & surjective | Invertible transforms |

In [ ]:
# Testing Injectivity (one-to-one)

def is_injective(func, domain):
    """Check if function is injective on given domain."""
    outputs = [func(x) for x in domain]
    return len(outputs) == len(set(outputs))

# f(x) = x² is NOT injective on ℝ (e.g., f(-2) = f(2) = 4)
domain_real = [-2, -1, 0, 1, 2]
print(f"f(x) = x² on {domain_real}")
print(f"Injective: {is_injective(lambda x: x**2, domain_real)}")

# f(x) = x² IS injective on non-negative reals
domain_nonneg = [0, 1, 2, 3, 4]
print(f"\nf(x) = x² on {domain_nonneg}")
print(f"Injective: {is_injective(lambda x: x**2, domain_nonneg)}")

In [ ]:
# Surjectivity and Bijectivity Examples

def is_surjective(func, domain, codomain):
    """Check if function covers all of codomain."""
    range_set = set(func(x) for x in domain)
    return codomain.issubset(range_set)

# Example: Mapping students to grades
students = ['Alice', 'Bob', 'Carol', 'Dave']
grades = {'Alice': 'A', 'Bob': 'B', 'Carol': 'A', 'Dave': 'C'}
get_grade = lambda s: grades[s]

codomain = {'A', 'B', 'C', 'D', 'F'}  # All possible grades
actual_range = set(grades.values())

print(f"Grades assigned: {actual_range}")
print(f"Surjective onto {codomain}? {actual_range == codomain}")
print(f"Injective? {len(students) == len(set(grades.values()))}")  # Alice & Carol both A

## 3. Function Composition

Given f: A → B and g: B → C, the composition (g ∘ f): A → C is:

**(g ∘ f)(x) = g(f(x))**

**Key insight**: Neural networks are function compositions!
- Layer 1: f₁(x)
- Layer 2: f₂(f₁(x))
- Layer n: fₙ(fₙ₋₁(...f₁(x)))

In [ ]:
# Function Composition

def compose(g, f):
    """Return composition g ∘ f."""
    return lambda x: g(f(x))

# Example: Neural network layer = linear + activation
def linear(x, w=2, b=1):
    return w * x + b

def relu(x):
    return np.maximum(0, x)

# Single layer: relu(linear(x))
layer = compose(relu, lambda x: linear(x))

x = np.array([-2, -1, 0, 1, 2])
print(f"Input x:      {x}")
print(f"Linear(x):    {linear(x)}")
print(f"ReLU(Lin(x)): {layer(x)}")

In [ ]:
# Multi-layer Composition (Deep Network)

from functools import reduce

def make_layer(w, b):
    """Create a layer: ReLU(wx + b)."""
    return lambda x: np.maximum(0, w * x + b)

# 3-layer network
layers = [
    make_layer(2, 1),   # Layer 1
    make_layer(-1, 3),  # Layer 2  
    make_layer(0.5, 0)  # Layer 3
]

def forward(x, layers):
    """Compose all layers: fₙ ∘ ... ∘ f₁."""
    return reduce(lambda acc, f: f(acc), layers, x)

x = 1.0
print(f"Input: {x}")
for i, layer in enumerate(layers):
    x = layer(x)
    print(f"After layer {i+1}: {x}")

## 4. Inverse Functions

For bijective f: A → B, the inverse f⁻¹: B → A satisfies:
- f⁻¹(f(x)) = x for all x ∈ A
- f(f⁻¹(y)) = y for all y ∈ B

**ML Applications:**
- Normalizing flows (invertible transforms)
- Encoder-decoder (approximate inverse)
- Log/exp transforms for numerical stability

In [ ]:
# Inverse Functions

# Sigmoid and its inverse (logit)
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def logit(p):
    """Inverse of sigmoid: logit(p) = log(p / (1-p))"""
    return np.log(p / (1 - p))

# Verify inverse property
x = np.array([-2, -1, 0, 1, 2])
print(f"x:               {x}")
print(f"sigmoid(x):      {sigmoid(x).round(4)}")
print(f"logit(sig(x)):   {logit(sigmoid(x)).round(4)}")
print(f"\n✓ logit(sigmoid(x)) = x")

## 5. ML Activation Functions

Activation functions introduce non-linearity, enabling neural networks to learn complex patterns.

| Function | Formula | Range | Use Case |
|----------|---------|-------|----------|
| Sigmoid | 1/(1+e⁻ˣ) | (0, 1) | Binary classification |
| Tanh | (eˣ-e⁻ˣ)/(eˣ+e⁻ˣ) | (-1, 1) | Hidden layers |
| ReLU | max(0, x) | [0, ∞) | Default choice |
| Softmax | eˣⁱ/Σeˣʲ | (0, 1), sum=1 | Multi-class |

In [ ]:
# Common Activation Functions

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def tanh(x):
    return np.tanh(x)

def relu(x):
    return np.maximum(0, x)

def leaky_relu(x, alpha=0.01):
    return np.where(x > 0, x, alpha * x)

def softmax(x):
    exp_x = np.exp(x - np.max(x))  # Numerical stability
    return exp_x / exp_x.sum()

# Compare on sample input
x = np.array([-2, -1, 0, 1, 2])
print(f"Input:      {x}")
print(f"Sigmoid:    {sigmoid(x).round(3)}")
print(f"Tanh:       {tanh(x).round(3)}")
print(f"ReLU:       {relu(x)}")
print(f"LeakyReLU:  {leaky_relu(x)}")

In [ ]:
# Softmax: Maps ℝⁿ → probability simplex

logits = np.array([2.0, 1.0, 0.1])  # Raw model outputs
probs = softmax(logits)

print(f"Logits: {logits}")
print(f"Softmax output: {probs.round(4)}")
print(f"Sum of probs: {probs.sum():.4f}")
print(f"\nProperties:")
print(f"  - All positive: {(probs > 0).all()}")
print(f"  - Sum to 1: {np.isclose(probs.sum(), 1)}")
print(f"  - Surjective onto simplex ✓")

## 6. Loss Functions as Mappings

Loss functions map (predictions, targets) → scalar:
- **L**: ℝⁿ × ℝⁿ → ℝ⁺

This scalar guides optimization (minimization).

In [ ]:
# Common Loss Functions

def mse_loss(y_pred, y_true):
    """Mean Squared Error: ℝⁿ × ℝⁿ → ℝ⁺"""
    return np.mean((y_pred - y_true) ** 2)

def cross_entropy_loss(probs, y_true_idx):
    """Cross-entropy: probability × label → ℝ⁺"""
    return -np.log(probs[y_true_idx] + 1e-10)

def binary_cross_entropy(y_pred, y_true):
    """BCE: [0,1] × {0,1} → ℝ⁺"""
    return -np.mean(y_true * np.log(y_pred + 1e-10) + 
                    (1 - y_true) * np.log(1 - y_pred + 1e-10))

# Examples
y_true = np.array([1.0, 2.0, 3.0])
y_pred = np.array([1.1, 2.2, 2.8])
print(f"MSE Loss: {mse_loss(y_pred, y_true):.4f}")

probs = softmax(np.array([2.0, 1.0, 0.5]))
print(f"Cross-Entropy (class 0): {cross_entropy_loss(probs, 0):.4f}")

## 7. Feature Transformations

Common transformations for data preprocessing:

| Transform | Formula | When to Use |
|-----------|---------|-------------|
| Z-score | (x - μ) / σ | Normalize to N(0,1) |
| Min-Max | (x - min) / (max - min) | Scale to [0, 1] |
| Log | log(x) | Reduce skewness |
| Box-Cox | (xᵏ - 1) / λ | Normalize distributions |

In [ ]:
# Feature Transformations

def z_score(x):
    """Standardize to mean=0, std=1"""
    return (x - np.mean(x)) / np.std(x)

def min_max_scale(x):
    """Scale to [0, 1]"""
    return (x - np.min(x)) / (np.max(x) - np.min(x))

def log_transform(x):
    """Log transform (handles zeros)"""
    return np.log1p(x)  # log(1 + x)

# Example: Skewed data
data = np.array([1, 2, 3, 10, 100, 1000])
print(f"Original:   {data}")
print(f"Z-score:    {z_score(data).round(2)}")
print(f"Min-Max:    {min_max_scale(data).round(2)}")
print(f"Log(1+x):   {log_transform(data).round(2)}")

## Summary

### Key Concepts:
| Concept | Definition | ML Relevance |
|---------|------------|--------------|
| Function | f: A → B mapping | Every ML model is a function |
| Injective | One-to-one | Embeddings preserve distinctness |
| Surjective | Onto | Output covers target space |
| Bijective | Invertible | Normalizing flows, autoencoders |
| Composition | g ∘ f | Neural network layers |
| Inverse | f⁻¹(f(x)) = x | Encoder-decoder pairs |

### Key Takeaways:
1. **Neural networks** are compositions of simple functions
2. **Activation functions** introduce non-linearity
3. **Loss functions** map predictions to optimization targets
4. **Transformations** prepare data for learning